# anyimage — BioImageViewer Example

This notebook demonstrates the core features of the `anyimage` package for visualizing biological images in Jupyter.

**Features covered:**
- Loading images with BioImage (TIFF, OME-Zarr)
- Displaying multi-dimensional images with BioImageViewer
- Adding mask overlays
- Using annotation tools (rectangles, polygons, points)
- Exporting annotations as DataFrames
- SAM (Segment Anything Model) integration

## 1. Imports

In [1]:
import pandas as pd
from bioio import BioImage
import bioio_tifffile
import bioio_ome_zarr
from anyimage import BioImageViewer

## 2. Load an image

Use `BioImage` to load images in TIFF or OME-Zarr format. Images are represented as 5D arrays: **(T, C, Z, Y, X)**.

In [11]:
# Load from OME-Zarr (lazy loading — efficient for large images)
img = BioImage("../image.zarr", reader=bioio_ome_zarr.Reader)
print("Image shape (TCZYX):", img.shape)

# Alternatively, load from TIFF:
# img = BioImage("image.tif", reader=bioio_tifffile.Reader)

Image shape (TCZYX): (10, 1, 3, 2048, 2048)


/var/home/maartenpaul/Documents/GitHub/anyimage/.venv/lib/python3.12/site-packages/zarr/core/metadata/v2.py:190: ZarrUserWarning: Found an empty list of filters in the array metadata document. This is contrary to the Zarr V2 specification, and will cause an error in the future. Use None (or Null in a JSON document) instead of an empty list of filters.
  warnings.warn(msg, ZarrUserWarning, stacklevel=1)


## 3. Display the image

Create a `BioImageViewer` widget and pass the `BioImage` object directly. This enables lazy loading and activates T/Z/C sliders when the image has multiple dimensions.

The viewer renders inline in Jupyter — just evaluate the `viewer` variable in a cell.

In [12]:
viewer = BioImageViewer()
viewer.set_image(img)
viewer

## 4. Add a mask overlay

Overlay segmentation masks on the image. Multiple masks can be added with different names, colors, and opacity.

In [13]:
img = BioImage("../image.tif", reader=bioio_tifffile.Reader)

viewer2 = BioImageViewer()
viewer2.set_image(img)
viewer2

mask = BioImage("../mask.tif", reader=bioio_tifffile.Reader)

mask_id = viewer2.add_mask(
    mask.data,
    name="Segmentation",
    color="#ff0000",
    opacity=0.5,
    contours_only=False
)
print("Added mask with ID:", mask_id)

# Add more masks with different colors:
# viewer.add_mask(nuclei_mask, name="Nuclei", color="#00ff00", opacity=0.3)
# viewer.add_mask(cell_mask, name="Cells", color="#0000ff", contours_only=True)

viewer2

Added mask with ID: mask_0_139800119120624


## 5. Annotation tools

Use the toolbar in the viewer to draw annotations:

| Tool | Shortcut | Description |
|------|----------|-------------|
| **Pan** | `P` | Navigate and zoom the image |
| **Select** | `V` | Click to select; `Delete` to remove |
| **Rectangle** | `R` | Click and drag to draw bounding boxes |
| **Polygon** | `G` | Click to add vertices; double-click or click near start to close |
| **Point** | `O` | Click to place point markers |

Use the **Layers** dropdown to toggle visibility, adjust opacity, and change mask colors.

## 6. Export annotations as DataFrames

After drawing annotations, access them as pandas DataFrames via the viewer properties.

> **Tip:** Re-run this cell after drawing annotations to see updated results.

In [ ]:
print("=== Rectangles (ROIs) ===")
display(viewer.rois_df)

print("\n=== Polygons ===")
display(viewer.polygons_df)

print("\n=== Points ===")
display(viewer.points_df)

## 7. Mask layer info

In [ ]:
print("Active mask IDs:", viewer.get_mask_ids())

# Update mask settings programmatically:
# viewer.update_mask_settings(mask_id, opacity=0.3, color="#00ff00")

# Remove a mask:
# viewer.remove_mask(mask_id)

# Clear all masks:
# viewer.clear_masks()

## 8. SAM — Segment Anything Model

Enable SAM to automatically generate segmentation masks from rectangle or point annotations.

**How to use:**
1. Select the **Rectangle** tool (`R`)
2. Draw a bounding box around an object
3. SAM will generate a segmentation mask automatically

**Available models:**
- `mobile_sam` — Fastest, ~40 MB (default)
- `fast_sam` — Fast, CNN-based
- `sam_b` — SAM base model
- `sam_l` — SAM large model

> **Note:** SAM requires PyTorch and the `sam` extra: `uv pip install -e ".[sam]"`

In [ ]:
img_sam = BioImage("fluocell.tif", reader=bioio_tifffile.Reader)

viewer_sam = BioImageViewer()
viewer_sam.set_image(img_sam.data)
viewer_sam.enable_sam(model_type="mobile_sam")
viewer_sam

In [ ]:
# After drawing rectangles, inspect the generated SAM masks and bounding boxes:
print("=== SAM Masks ===")
print("Mask IDs:", viewer_sam.get_mask_ids())

print("\n=== Bounding Boxes (ROIs) ===")
display(viewer_sam.rois_df)